# IMC2025 — Modular SfM Pipeline

**Task:** given folders of unordered images (one folder per *dataset*), recover a camera pose for every image that can be registered and group the images into *scenes*. Images that cannot be registered are labelled `outliers`.

## Current implementation

| # | Step | Current approach | Main output variable |
|---|------|------------------|----------------------|
| 1 | Discover images | `torchvision` `ImageFolder` index | `image_paths` |
| 2 | Global embeddings | DINOv2 ViT-S/14 (CLS token) | `embeddings` |
| 3 | Pair shortlisting | k-NN (cosine) inside each dataset | `pairs` |
| 4 | Local features | ALIKED or SuperPoint, set by `LIGHTGLUE_FEATURES` (via the `lightglue` package) | `features` |
| 5 | Local matching | LightGlue | `match_records` |
| 6 | Geometric verification | OpenCV fundamental-matrix PROSAC (algorithm and model switchable, see 0.2) | `ransac_results` |
| 7 | Pair filtering | thresholds on matches / inliers / inlier ratio | `filtered_pairs` |
| 8 | Image graph | `networkx` graph with keypoints + inlier matches | `image_graph` |
| 9 | Component inspection *(optional)* | connected components | `components` |
| 10 | Reconstruction | `pycolmap` incremental SfM per dataset | `db_paths`, `reconstructions`, `rows` |
| 11 | Submission | CSV in the IMC format, preserving Kaggle's supplied `image_id` | `submission_df` |
| 12 | Evaluation | official IMC metric on the train split | `final_score` |

**Ideas from the original notes, not implemented yet:** HDBSCAN (cosine) clustering before pairing, SuperGlue, MASt3R, and a second "fine" feature pass (the `FINE_*` hyperparameters exist but are unused).

## How to use this notebook

* Run top to bottom. The first code cell holds all imports, the second all hyperparameters.
* Every pipeline cell starts with an **INPUTS / OUTPUTS** header that states variable names, types and shapes. To try a different approach, replace the body of one step cell and keep its output contract; the later steps will keep working.

## Conventions used throughout

* `dataset` = folder name under `train/` (e.g. `'ETs'`). `image` = file name inside that folder (the code assumes flat folders).
* Keypoints are `(x, y)` pixel coordinates in the **original-resolution** image, using the pixel-centre-is-integer convention (the centre of the top-left pixel is `(0, 0)`). COLMAP uses centre `(0.5, 0.5)`, hence the `+0.5` in step 10.
* A *match* is an index pair `(i, j)`: keypoint `i` of image 0 corresponds to keypoint `j` of image 1.

## Changes in this version (hidden-test robustness)

| # | Problem | Fix | Where |
|---|---------|-----|-------|
| 1 | Steps 4b, 10b and 10c read images from `TRAIN_DIR`, while step 1 listed `TEST_DIR`. Works on the visible sample only because `ETs` and `stairs` also exist under `train/`; on the hidden test set the folders are not there. | One `IMAGE_DIR = TEST_DIR` used everywhere. | 0.2, 1, 4b, 10b, 10c |
| 2 | Step 2c opened hard-coded example images and crashed when they were missing. | The cell skips itself when the files are not on disk. | 2c |
| 3 | The images that appear in the output came from the candidate pairs, so images without a pair (e.g. a single-image dataset) had no row. | Rows are built from `image_paths` (everything on disk). | 10d, 11 |
| 4 | COLMAP can register one image in several models, which produced duplicate rows. | An image is kept in its largest model only. | 10d |
| 5 | One failing dataset (or a dataset without verified pairs) could kill the whole run. | Per-dataset `try/except` with fallback to `outliers`; COLMAP is skipped when there are no verified pairs; the summary prints no longer crash on empty results. | 4b, 5b, 10b, 10c, 10d |
| 6 | No check of the final file. | `validate_and_repair` guarantees each sample-submission image appears once with its exact Kaggle `image_id` before saving. | 11 |
| 7 | Relative output paths and leftover scratch folders. | Absolute `/kaggle/working/...` paths; `colmap_work/` and `torch_cache/` are deleted at the end. | 0.2, cleanup |

Not changed: there is still no runtime budget, so a much larger hidden test set could run into the time limit.


## 0. Setup

### 0.1 Dependencies
All imports live in this single cell. Nothing else in the notebook imports anything.

In [63]:
from pathlib import Path
import sys

# Dataset: imc-model-assets
KAGGLE_MODELS_DIR = Path('/kaggle/input/datasets/tudorluchiean/imc-model-assets/KaggleModels')
DINOV2_DIR = KAGGLE_MODELS_DIR / 'dinov2'
LIGHTGLUE_DIR = KAGGLE_MODELS_DIR / 'LightGlue'
CHECKPOINT_DIR = KAGGLE_MODELS_DIR / 'checkpoints'

# Dataset: imc-new-models
NEW_MODELS_DIR = Path('/kaggle/input/datasets/tudorluchiean/imc-new-models')
DINOV3_REPO_ROOT = NEW_MODELS_DIR / 'dinov3'
DINOV3_CHECKPOINT = NEW_MODELS_DIR / 'dinov3_vits16_pretrain_lvd1689m-08c60483.pth'
SIGLIP_DIR = NEW_MODELS_DIR / 'siglip-base-patch16-224'

# Dataset: aliked
ALIKED_DIR = Path('/kaggle/input/datasets/tudorluchiean/aliked-checkpoints')
ALIKED_CHECKPOINT = ALIKED_DIR / 'aliked-n16.pth'
ALIKED_LIGHTGLUE_CHECKPOINT = ALIKED_DIR / 'aliked_lightglue_v0-1_arxiv.pth'

# The DINOv3 repository is nested in the attached dataset. Find the actual
# repository directory: its direct contents must include hubconf.py.
dinov3_hubconf_files = sorted(DINOV3_REPO_ROOT.rglob('hubconf.py')) if DINOV3_REPO_ROOT.exists() else []
if len(dinov3_hubconf_files) != 1:
    raise FileNotFoundError(
        f'Expected exactly one DINOv3 hubconf.py under {DINOV3_REPO_ROOT}, '
        f'found {len(dinov3_hubconf_files)}: {dinov3_hubconf_files}'
    )
DINOV3_DIR = dinov3_hubconf_files[0].parent

sys.path.insert(0, str(LIGHTGLUE_DIR))

required_dependency_paths = {
    'DINOv2 repository': DINOV2_DIR / 'hubconf.py',
    'DINOv2 checkpoint': CHECKPOINT_DIR / 'dinov2_vits14_pretrain.pth',
    'LightGlue package': LIGHTGLUE_DIR / 'lightglue',
    'DINOv3 repository': DINOV3_DIR / 'hubconf.py',
    'DINOv3 checkpoint': DINOV3_CHECKPOINT,
    'SigLIP config': SIGLIP_DIR / 'config.json',
    'ALIKED checkpoint': ALIKED_CHECKPOINT,
    'ALIKED-LightGlue checkpoint': ALIKED_LIGHTGLUE_CHECKPOINT,
    'SuperPoint checkpoint': CHECKPOINT_DIR / 'superpoint_v1.pth',
    'SuperPoint-LightGlue checkpoint': CHECKPOINT_DIR / 'superpoint_lightglue.pth',
}
for dependency, path in required_dependency_paths.items():
    print(f"{'OK' if path.exists() else 'MISSING'} | {dependency}: {path}")

missing_dependencies = [
    f'{name}: {path}' for name, path in required_dependency_paths.items()
    if not path.exists()
]
if missing_dependencies:
    raise FileNotFoundError(
        'Missing dependencies. Confirm that all three datasets are attached and '
        'the files have the expected names:\n  - ' + '\n  - '.join(missing_dependencies)
    )


OK | DINOv2 repository: /kaggle/input/datasets/tudorluchiean/imc-model-assets/KaggleModels/dinov2/hubconf.py
OK | DINOv2 checkpoint: /kaggle/input/datasets/tudorluchiean/imc-model-assets/KaggleModels/checkpoints/dinov2_vits14_pretrain.pth
OK | LightGlue package: /kaggle/input/datasets/tudorluchiean/imc-model-assets/KaggleModels/LightGlue/lightglue
OK | DINOv3 repository: /kaggle/input/datasets/tudorluchiean/imc-new-models/dinov3/hubconf.py
OK | DINOv3 checkpoint: /kaggle/input/datasets/tudorluchiean/imc-new-models/dinov3_vits16_pretrain_lvd1689m-08c60483.pth
OK | SigLIP config: /kaggle/input/datasets/tudorluchiean/imc-new-models/siglip-base-patch16-224/config.json
OK | ALIKED checkpoint: /kaggle/input/datasets/tudorluchiean/aliked-checkpoints/aliked-n16.pth
OK | ALIKED-LightGlue checkpoint: /kaggle/input/datasets/tudorluchiean/aliked-checkpoints/aliked_lightglue_v0-1_arxiv.pth
OK | SuperPoint checkpoint: /kaggle/input/datasets/tudorluchiean/imc-model-assets/KaggleModels/checkpoints/sup

In [64]:
import shutil
from pathlib import Path
import torch

TORCH_HOME = Path('/kaggle/working/torch_cache')
CHECKPOINT_CACHE = TORCH_HOME / 'hub' / 'checkpoints'
CHECKPOINT_CACHE.mkdir(parents=True, exist_ok=True)

# Source path -> exact filename expected by the LightGlue package.
checkpoint_sources = {
    'superpoint_v1.pth': CHECKPOINT_DIR / 'superpoint_v1.pth',
    'superpoint_lightglue_v0-1_arxiv.pth': CHECKPOINT_DIR / 'superpoint_lightglue.pth',
    'aliked-n16.pth': ALIKED_CHECKPOINT,
    'aliked_lightglue_v0-1_arxiv.pth': ALIKED_LIGHTGLUE_CHECKPOINT,
}
for cache_name, source in checkpoint_sources.items():
    if not source.is_file():
        raise FileNotFoundError(f'Missing local-feature checkpoint: {source}')
    destination = CHECKPOINT_CACHE / cache_name
    shutil.copy2(source, destination)
    print(f'Cached {cache_name} from {source}')

# Set the cache before constructing ALIKED, SuperPoint or LightGlue.
torch.hub.set_dir(str(TORCH_HOME / 'hub'))


Cached superpoint_v1.pth from /kaggle/input/datasets/tudorluchiean/imc-model-assets/KaggleModels/checkpoints/superpoint_v1.pth
Cached superpoint_lightglue_v0-1_arxiv.pth from /kaggle/input/datasets/tudorluchiean/imc-model-assets/KaggleModels/checkpoints/superpoint_lightglue.pth
Cached aliked-n16.pth from /kaggle/input/datasets/tudorluchiean/aliked-checkpoints/aliked-n16.pth
Cached aliked_lightglue_v0-1_arxiv.pth from /kaggle/input/datasets/tudorluchiean/aliked-checkpoints/aliked_lightglue_v0-1_arxiv.pth


In [65]:
import subprocess
import sys
from pathlib import Path

# pycolmap is not in the Kaggle image and the notebook runs offline, so install it from the attached wheel
#wheels = sorted(Path("/kaggle/input").rglob("pycolmap-*.whl"))
#assert wheels, "Missing pycolmap wheel in /kaggle/input"

#subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", str(wheels[0])], check=True)
#print(f"Installed: {wheels[0].name}")

In [66]:
# ── Standard library ────────────────────────────────────────────────
import shutil
import sys
import time
import traceback
from pathlib import Path

# ── Numerics / tabular ──────────────────────────────────────────────
import numpy as np
import pandas as pd
from tqdm import tqdm

# ── Deep learning: embeddings, local features, matching ─────────────
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
from lightglue import ALIKED, SuperPoint, LightGlue
from lightglue.utils import load_image, rbd

# ── Classical CV / ML ───────────────────────────────────────────────
import cv2
from sklearn.neighbors import NearestNeighbors

# ── Graph + Structure-from-Motion ───────────────────────────────────
import networkx as nx
import pycolmap

# ── Optional embedding backbone (only used when 'siglip' is in EMBED_MODELS) ─────────
try:
    from transformers import SiglipVisionModel
except ImportError:
    SiglipVisionModel = None


### 0.2 Hyperparameters and paths
Every tunable value of the pipeline lives here, grouped by step. Change values here and re-run from the step you changed.

**Run a different variant by editing only the `VARIANT SWITCHES` block at the top** (embeddings, shortlisting, geometric verification).


In [67]:
# ═══ VARIANT SWITCHES ═══════════════════════════════════════════════
# Edit only this block to run a variant. Defaults reproduce the previous behaviour, except GEOM_* (new default: PROSAC + fundamental).

# -- Step 2 · embeddings
EMBED_MODELS        = ('dinov2','dinov3')             # any combination of 'dinov2', 'dinov3', 'siglip'
EMBED_WEIGHTS       = {'dinov2': 1.0, 'dinov3': 1.0, 'siglip': 1.0}   # weight of each model's L2-normalised block in the concatenation (similarity scales with weight^2)
EMBED_IMAGE_SIZE    = {'dinov2': (518, 518), 'dinov3': (512, 512), 'siglip': (224, 224)}   # (H, W) per model, or one (H, W) for all; dinov2 -> multiple of 14, dinov3 -> multiple of 16, siglip -> checkpoint's native size
EMBED_POOLING       = 'cls'                   # 'cls' (siglip: its pooling head) | 'mean' (mean of patch tokens) | 'gem' (generalised mean of patch tokens)
EMBED_GEM_P         = 3.0                     # GeM exponent, only used when EMBED_POOLING = 'gem'
EMBED_FLIP_TTA      = False                   # True = average the embedding of each image and of its horizontal flip
DINOV3_DIR          = DINOV3_DIR             # resolved in the dependency setup cell
DINOV3_NAME         = 'dinov3_vits16'         # DINOv3 hub entry point (ViT-S/16 -> 384-dim)
DINOV3_WEIGHTS_GLOB = 'dinov3_vits16_pretrain*.pth'   # retained for logging/reference; checkpoint is loaded from DINOV3_CHECKPOINT
SIGLIP_DIR          = SIGLIP_DIR             # local HF checkpoint path resolved in dependency setup

# -- Step 3 · pair shortlisting
K_NEIGHBORS         = 35                      # nearest neighbours per image in 'combined' mode (self excluded)
KNN_METRIC          = 'manhattan'                # 'cosine' | 'euclidean' | 'manhattan'
SHORTLIST_MODE      = 'combined'              # 'combined' (k-NN on the combined embedding) | 'per_model_union' (top K_PER_MODEL per model, union)
K_PER_MODEL         = 30                      # neighbours per image and per model in 'per_model_union' mode
QUERY_EXPANSION     = False                   # True = merge each image's QE_TOP_K nearest-neighbour embeddings into its own, then rerun the k-NN
QE_TOP_K            = 0                       # neighbours merged into each image by query expansion

# -- Step 6 · geometric verification
GEOM_ALGORITHM      = 'magsac++'                # 'ransac' | 'lo_ransac' | 'magsac++' | 'prosac' | 'usac'
GEOM_MODEL          = 'fundamental'           # 'fundamental' | 'essential' (intrinsics guessed from image size) | 'homography'
GEOM_FOCAL_FACTOR   = 1.2                     # essential only: focal length guess = factor * max(image width, height)

# ═══ PATHS ══════════════════════════════════════════════════════════
DATA_ROOT            = Path('/kaggle/input/competitions/image-matching-challenge-2025')
TRAIN_DIR            = DATA_ROOT / 'train'                 # TRAIN_DIR/<dataset>/<image files>
TEST_DIR            = DATA_ROOT / 'test'
IMAGE_DIR            = TEST_DIR                            # the images to reconstruct. ALWAYS the test folder: on the hidden
                                                           # rerun it holds the hidden datasets, which do NOT exist under train/
OUTPUT_DIR           = Path('/kaggle/working')             # everything written by the notebook lives here
TRAIN_LABELS_CSV     = DATA_ROOT / 'train_labels.csv'
TRAIN_THRESHOLDS_CSV = DATA_ROOT / 'train_thresholds.csv'
WORK_DIR             = OUTPUT_DIR / 'colmap_work'                 # COLMAP databases + sparse models (wiped per dataset)
SUBMISSION_CSV       = OUTPUT_DIR / 'submission.csv'
SAMPLE_SUBMISSION_CSV = DATA_ROOT / 'sample_submission.csv'  # authoritative (image_id, dataset, image) mapping
GT_SUBSET_CSV        = OUTPUT_DIR / 'gt_subset.csv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Three images for the optional sanity check in step 2c: [scene A, scene A (same), scene B]
EXAMPLE_IMAGES = [
    DATA_ROOT / 'test' / 'stairs' / 'stairs_split_1_1710453576271.png',
    DATA_ROOT / 'test' / 'stairs' / 'stairs_split_1_1710453601885.png',
    DATA_ROOT / 'test' / 'ETs'    / 'another_et_another_et001.png',
]

# ═══ DEVICE ═════════════════════════════════════════════════════════
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

# ═══ STEP 2 · GLOBAL EMBEDDINGS ═════════════════════════════════════
EMBED_MODEL_REPO  = 'facebookresearch/dinov2'
EMBED_MODEL_NAME  = 'dinov2_vits14'               # ViT-S/14 -> 384-dim embedding   (EMBED_IMAGE_SIZE: see VARIANT SWITCHES)
EMBED_NORM_MEAN   = (0.485, 0.456, 0.406)         # ImageNet statistics (DINO models)
EMBED_NORM_STD    = (0.229, 0.224, 0.225)
EMBED_BATCH_SIZE  = 64

# ═══ STEP 3 · PAIR SHORTLISTING ═════════════════════════════════════
# K_NEIGHBORS, KNN_METRIC and the other shortlisting options: see VARIANT SWITCHES

# ═══ STEP 4/5 · LOCAL FEATURES + MATCHING ═══════════════════════════
COARSE_MAX_KEYPOINTS = 2048                        # keypoint budget per image
COARSE_RESIZE        = 1024                     # longer image side fed to the extractor
FINE_MAX_KEYPOINTS   = 2048                       # UNUSED so far (reserved for a second, finer pass)
FINE_RESIZE          = 1024                       # UNUSED so far
LIGHTGLUE_FEATURES   = ('aliked', 'superpoint')                # extractor + LightGlue weights: 'aliked' or 'superpoint'
ALIKED_MODEL_NAME    = 'aliked-n16'               # ALIKED variant, 128-dim descriptors
ALIKED_THRESHOLD     = 0.01                       # min detection score; the library default 0.2 leaves far fewer keypoints
ALIKED_MAX_KEYPOINTS = 2048

ROTATION_RETRY_BELOW = 50                         # raw matches; weaker pairs are retried with image b rotated (0 = no retries)
ROTATION_RETRY_FEATURES = ('aliked',)             # extractors used for the retries
ROTATION_MIN_GAIN    = 2.0                        # a rotation must have this many times the raw matches found at 0 deg

TILE_ENABLE          = True                       # second matching pass on high-res crops of each image (False = off)
TILE_GRID            = (2, 2)                      # crops per image as (rows, cols); (2, 2) = 4 tiles
TILE_OVERLAP         = 0.10                        # fraction each tile overextends into its neighbours, so border keypoints survive
TILE_RESIZE          = 1200                       # longer side each crop is resized to before extraction
TILE_FEATURES        = ('aliked',)                # extractors used for the tile pass (must be a subset of LIGHTGLUE_FEATURES)
TILE_ABOVE           = 15                         # only pairs with at least this many coarse matches are tiled (skips weak/unrelated pairs)
TILE_WITH_FULL       = True                       # also match each tile against the other image's full view (cross-scale); False = tiles vs tiles only
# ═══ STEP 6 · GEOMETRIC VERIFICATION (RANSAC) ═══════════════════════
RANSAC_THRESHOLD     = 2                      # px, defined at the resized scale (multiplied by the pair's `scale`)
RANSAC_CONFIDENCE    = 0.999
RANSAC_MIN_MATCHES   = 8                          # fewer matches than this -> no geometric model is estimated
VERIFIED_MIN_INLIERS = 8                          # sets the `verified` flag only (not used for filtering)

# ═══ STEP 7 · PAIR FILTERING ════════════════════════════════════════
MIN_MATCHES      = 15
MIN_INLIERS      = 15
MIN_INLIER_RATIO = 0.10

# ═══ STEP 8 · IMAGE GRAPH ═══════════════════════════════════════════
KEYPOINT_CONSISTENCY_ATOL = 1e-4                  # same image must yield identical keypoints in every pair

# ═══ STEP 10 · COLMAP ═══════════════════════════════════════════════
COLMAP_CAMERA_MODE              = pycolmap.CameraMode.AUTO
COLMAP_KEYPOINT_OFFSET          = 0.5             # shift from pixel-centre=0 to COLMAP's pixel-centre=0.5
MAPPER_MIN_MODEL_SIZE           = 3               # min #registered images for a model to be kept
MAPPER_INIT_MIN_NUM_INLIERS     = 15              # min inliers for the initial image pair
MAPPER_ABS_POSE_MIN_NUM_INLIERS = 10              # min inliers to register a new image

---
## Step 1 · Discover images and load submission IDs

**What it does:** walks `TEST_DIR` and builds an index of which image files belong to which dataset. It also reads Kaggle's supplied `sample_submission.csv`, whose `image_id` values are the authoritative row identifiers for the current test set. No pixels are loaded here.

**Why separate:** later steps (embedding, feature extraction, COLMAP) all need the same ordered file list. Keeping the index in one place guarantees that row `i` of an embedding matrix always refers to `image_paths[name][i]`. The sample-submission check independently confirms that the discovered test images and their required IDs agree.

**Replace with:** a plain `pathlib` glob, a CSV of test-set images, etc. Only the output contract matters.


In [68]:
# ══════════════════════════════════════════════════════════════════════
# STEP 1 · DISCOVER IMAGES + LOAD KAGGLE IMAGE IDs
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   IMAGE_DIR                Path   test/<dataset>/<image files>
#   SAMPLE_SUBMISSION_CSV    Path   Kaggle's current test-set row identifiers
# OUTPUTS
#   image_paths              dict[str, list[Path]]
#                            dataset name -> image paths, sorted by file name
#   sample_submission_index  pd.DataFrame[image_id, dataset, image]
#                            authoritative mapping supplied by Kaggle
#
# `image_id` is deliberately read from sample_submission.csv. It is a unique
# identifier assigned by the competition; it is not a sequential row number or
# a value that should be reconstructed from a filename.
# ══════════════════════════════════════════════════════════════════════

def discover_images(image_dir):
    folder = ImageFolder(str(image_dir))        # no transform -> only builds the file index, loads no images
    return {
        name: [Path(p) for p, label in folder.samples if label == idx]
        for name, idx in folder.class_to_idx.items()
    }


def load_submission_index(sample_submission_csv):
    '''Load the immutable ID mapping that Kaggle expects in this submission.'''
    required = {'image_id', 'dataset', 'image'}
    sample = pd.read_csv(sample_submission_csv, dtype='string', keep_default_na=False)
    missing_columns = required - set(sample.columns)
    if missing_columns:
        raise ValueError(
            f'{sample_submission_csv} is missing required column(s): '
            f'{sorted(missing_columns)}'
        )

    index = sample.loc[:, ['image_id', 'dataset', 'image']].copy()
    if (index == '').any().any():
        raise ValueError('sample_submission.csv contains an empty image_id, dataset, or image value')
    if index.duplicated(['dataset', 'image']).any():
        raise ValueError('sample_submission.csv has duplicate (dataset, image) keys')
    if index['image_id'].duplicated().any():
        raise ValueError('sample_submission.csv has duplicate image_id values')
    return index


image_paths = discover_images(IMAGE_DIR)
sample_submission_index = load_submission_index(SAMPLE_SUBMISSION_CSV)

# The sample submission—not the discovery order—is the independent source of
# truth for which rows must be present and which image_id each row must retain.
discovered_keys = {(name, p.name) for name, paths in image_paths.items() for p in paths}
expected_keys = set(zip(sample_submission_index['dataset'], sample_submission_index['image']))
if discovered_keys != expected_keys:
    missing_on_disk = sorted(expected_keys - discovered_keys)
    missing_in_sample = sorted(discovered_keys - expected_keys)

    print("WARNING: Test-image discovery differs from sample_submission.csv.")
    print(f"Images expected by Kaggle: {len(expected_keys)}")
    print(f"Images available on disk: {len(discovered_keys)}")
    print(f"Images missing on disk: {len(missing_on_disk)}")
    print(f"Images absent from sample submission: {len(missing_in_sample)}")

    if missing_on_disk:
        print("Examples of missing images:", missing_on_disk[:5])

    if missing_in_sample:
        print("Examples of unexpected images:", missing_in_sample[:5])
for name, paths in image_paths.items():
    print(f'{name:<30} {len(paths):>5} images')
print(f'-> {len(image_paths)} datasets, {sum(map(len, image_paths.values()))} images in total')
print(f'-> {len(sample_submission_index)} exact image_id values loaded from {SAMPLE_SUBMISSION_CSV.name}')


Images expected by Kaggle: 1945
Images available on disk: 73
Images missing on disk: 1872
Images absent from sample submission: 0
Examples of missing images: [('amy_gardens', 'peach_0000.png'), ('amy_gardens', 'peach_0001.png'), ('amy_gardens', 'peach_0002.png'), ('amy_gardens', 'peach_0003.png'), ('amy_gardens', 'peach_0004.png')]
ETs                               22 images
stairs                            51 images
-> 2 datasets, 73 images in total
-> 1945 exact image_id values loaded from sample_submission.csv


### Run config
Prints every variant switch so each run is documented.


In [69]:
# ══════════════════════════════════════════════════════════════════════
# RUN CONFIG · PRINT ALL VARIANT SWITCHES
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   the VARIANT SWITCHES block of cell 0.2
# OUTPUTS
#   (none, printed summary only)
# ══════════════════════════════════════════════════════════════════════

RUN_CONFIG_KEYS = (
    'EMBED_MODELS', 'EMBED_WEIGHTS', 'EMBED_IMAGE_SIZE', 'EMBED_POOLING', 'EMBED_GEM_P', 'EMBED_FLIP_TTA',
    'K_NEIGHBORS', 'KNN_METRIC', 'SHORTLIST_MODE', 'K_PER_MODEL', 'QUERY_EXPANSION', 'QE_TOP_K',
    'GEOM_ALGORITHM', 'GEOM_MODEL', 'GEOM_FOCAL_FACTOR',
)

print('RUN CONFIG')
for key in RUN_CONFIG_KEYS:
    print(f'  {key:<18} = {globals()[key]}')


RUN CONFIG
  EMBED_MODELS       = ('dinov2', 'dinov3')
  EMBED_WEIGHTS      = {'dinov2': 1.0, 'dinov3': 1.0, 'siglip': 1.0}
  EMBED_IMAGE_SIZE   = {'dinov2': (518, 518), 'dinov3': (512, 512), 'siglip': (224, 224)}
  EMBED_POOLING      = cls
  EMBED_GEM_P        = 3.0
  EMBED_FLIP_TTA     = False
  K_NEIGHBORS        = 35
  KNN_METRIC         = cosine
  SHORTLIST_MODE     = combined
  K_PER_MODEL        = 30
  QUERY_EXPANSION    = False
  QE_TOP_K           = 3
  GEOM_ALGORITHM     = prosac
  GEOM_MODEL         = fundamental
  GEOM_FOCAL_FACTOR  = 1.2


---
## Step 2 · Global image embeddings (DINOv2)

**What it does:** turns every image into one fixed-length vector that summarises its overall appearance. Images of the same scene end up close together (high cosine similarity). These vectors are used in step 3 to decide *which images are worth matching*, which avoids matching all `N²` pairs.

DINOv2 reference: https://github.com/facebookresearch/dinov2

### 2a · Load the embedding model and its preprocessing
Kept separate from the inference cell so the model/transform can be swapped (another DINOv2 size, CLIP, DINOv3, ...) without touching the loop.

**Variants:** `EMBED_MODELS`, `EMBED_WEIGHTS`, `EMBED_POOLING`, `EMBED_FLIP_TTA` (0.2, *VARIANT SWITCHES*) select DINOv2 / DINOv3 / SigLIP, their weighting, pooling and flip-TTA. Default = DINOv2 CLS token.


In [70]:
# ══════════════════════════════════════════════════════════════════════
# STEP 2a · LOAD EMBEDDING MODELS + TRANSFORMS
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   EMBED_MODELS, EMBED_WEIGHTS, EMBED_IMAGE_SIZE, EMBED_POOLING, EMBED_GEM_P, EMBED_FLIP_TTA   (VARIANT SWITCHES)
#   EMBED_MODEL_NAME, EMBED_NORM_MEAN/STD   DINOv2 hub name + ImageNet statistics
#   DINOV2_DIR, DINOV3_DIR, DINOV3_NAME, DINOV3_WEIGHTS_GLOB, SIGLIP_DIR, CHECKPOINT_DIR, DEVICE
# OUTPUTS
#   embed_models      dict[str, Embedder]   model name -> callable: Tensor [B, 3, H, W] -> Tensor [B, D_m]
#                     L2-normalised, pooling and optional flip-TTA already applied (D_m = 384 for dinov2_vits14)
#   embed_transform   callable: PIL.Image (RGB) -> dict[str, Tensor [3, H, W]]   one resized + normalised tensor per model
#   combine_embeddings(parts)   dict[model -> Tensor [N, D_m]] -> Tensor [N, D_total]   weighted concatenation
#   embed_image(img)  callable: PIL.Image (RGB) -> Tensor [1, D_total]   combined embedding of one image (used by 2c)
# ══════════════════════════════════════════════════════════════════════

EMBED_KINDS = ('dinov2', 'dinov3', 'siglip')
embed_model_names = (EMBED_MODELS,) if isinstance(EMBED_MODELS, str) else tuple(dict.fromkeys(EMBED_MODELS))
if not embed_model_names or any(m not in EMBED_KINDS for m in embed_model_names):
    raise ValueError(f'EMBED_MODELS must be a non-empty combination of {EMBED_KINDS}, got {EMBED_MODELS!r}')
if EMBED_POOLING not in ('cls', 'mean', 'gem'):
    raise ValueError(f"EMBED_POOLING must be 'cls', 'mean' or 'gem', got {EMBED_POOLING!r}")
for m in embed_model_names:
    if m not in EMBED_WEIGHTS:
        raise ValueError(f'EMBED_WEIGHTS has no entry for {m!r}')


class Embedder:
    """One backbone + pooling (+ flip TTA). Call: Tensor [B, 3, H, W] -> L2-normalised Tensor [B, D]."""
    def __init__(self, kind, model, pooling, flip_tta, gem_p):
        self.kind, self.model, self.pooling, self.flip_tta, self.gem_p = kind, model, pooling, flip_tta, gem_p

    def _pool(self, x):
        if self.kind == 'siglip':
            out = self.model(pixel_values=x)
            cls, tokens = out.pooler_output, out.last_hidden_state      # SigLIP has no CLS token: its pooling head plays that role
        elif self.pooling == 'cls':
            return self.model(x)                                        # DINO CLS token, exactly as before
        else:
            cls, tokens = None, self.model.forward_features(x)['x_norm_patchtokens']
        if self.pooling == 'cls':
            if cls is None:
                raise ValueError("This SigLIP checkpoint has no pooling head; use EMBED_POOLING = 'mean' or 'gem'")
            return cls
        if self.pooling == 'mean':
            return tokens.mean(dim=1)
        return tokens.clamp(min=1e-6).pow(self.gem_p).mean(dim=1).pow(1.0 / self.gem_p)   # GeM; clamp because ViT tokens can be negative

    @torch.no_grad()
    def __call__(self, x):
        e = F.normalize(self._pool(x), dim=1)
        if self.flip_tta:                                               # mean of image and horizontal flip (flip = width axis)
            e = F.normalize(e + F.normalize(self._pool(torch.flip(x, dims=[3])), dim=1), dim=1)
        return e


def load_embedder(kind):
    if kind == 'dinov2':
        model = torch.hub.load(
            str(DINOV2_DIR),
            EMBED_MODEL_NAME,
            source='local',
            weights=str(CHECKPOINT_DIR / 'dinov2_vits14_pretrain.pth')
        )
    elif kind == 'dinov3':
        if not (DINOV3_DIR / 'hubconf.py').exists() or not DINOV3_CHECKPOINT.is_file():
            raise FileNotFoundError(
                f"'dinov3' is in EMBED_MODELS but its files are missing: need "
                f"{DINOV3_DIR / 'hubconf.py'} and {DINOV3_CHECKPOINT}. "
                "Attach the DINOv3 repository and checkpoint or remove 'dinov3' from EMBED_MODELS.")
        model = torch.hub.load(str(DINOV3_DIR), DINOV3_NAME, source='local', weights=str(DINOV3_CHECKPOINT))
    else:  # 'siglip'
        if SiglipVisionModel is None:
            raise ImportError("'siglip' is in EMBED_MODELS but the `transformers` package is not installed")
        if not (SIGLIP_DIR / 'config.json').exists():
            raise FileNotFoundError(
                f"'siglip' is in EMBED_MODELS but {SIGLIP_DIR / 'config.json'} is missing. Attach a local Hugging Face "
                f"SigLIP folder (config + weights) at SIGLIP_DIR or remove 'siglip' from EMBED_MODELS.")
        model = SiglipVisionModel.from_pretrained(str(SIGLIP_DIR), local_files_only=True).float()
    return Embedder(kind, model.eval().to(DEVICE), EMBED_POOLING, EMBED_FLIP_TTA, EMBED_GEM_P)


PATCH_SIZE = {'dinov2': 14, 'dinov3': 16}


def make_transform(kind):
    size = EMBED_IMAGE_SIZE[kind] if isinstance(EMBED_IMAGE_SIZE, dict) else EMBED_IMAGE_SIZE
    if kind in PATCH_SIZE:                                              # DINO needs multiples of the patch size: round down
        p = PATCH_SIZE[kind]
        size = tuple(max(p, s // p * p) for s in size)
    if kind == 'siglip':
        mean = std = (0.5, 0.5, 0.5)
        interpolation = T.InterpolationMode.BICUBIC
    else:
        mean, std = EMBED_NORM_MEAN, EMBED_NORM_STD
        interpolation = T.InterpolationMode.BILINEAR
    return T.Compose([
        T.Resize(tuple(size), interpolation=interpolation),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])


embed_models = {m: load_embedder(m) for m in embed_model_names}
_transforms = {m: make_transform(m) for m in embed_models}


def embed_transform(img):
    return {m: transform(img) for m, transform in _transforms.items()}


def combine_embeddings(parts):
    return torch.cat([EMBED_WEIGHTS[m] * parts[m] for m in embed_models], dim=1)


@torch.no_grad()
def embed_image(img):
    x = embed_transform(img)
    return combine_embeddings({m: embed_models[m](x[m].unsqueeze(0).to(DEVICE)).cpu() for m in embed_models})


### 2b · Compute embeddings for every image
Runs the model over each dataset in batches. Output rows keep the order of `image_paths`.

In [71]:
# ══════════════════════════════════════════════════════════════════════
# STEP 2b · COMPUTE EMBEDDINGS
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   image_paths       dict[str, list[Path]]   from step 1
#   embed_models      dict[str, Embedder]     from step 2a
#   embed_transform   PIL.Image -> dict[str, Tensor[3,H,W]]   from step 2a
#   combine_embeddings                        from step 2a
#   EMBED_BATCH_SIZE, DEVICE
# OUTPUTS
#   embeddings        dict[str, Tensor]
#                     dataset name -> float32 CPU tensor of shape [N_images, D_total]
#                     row i is the embedding of image_paths[name][i]
#                     weighted concatenation of the per-model L2-normalised embeddings
#                     (D_total = 384 for the default dinov2 only)
#   embeddings_per_model   dict[str, dict[str, Tensor]]
#                     model name -> dataset name -> float32 CPU tensor [N_images, D_m], L2-normalised, unweighted
# ══════════════════════════════════════════════════════════════════════

class ImageListDataset(Dataset):
    """Loads images from a list of paths and applies `transform` (here: one tensor [3, H, W] per model)."""
    def __init__(self, paths, transform):
        self.paths, self.transform = paths, transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.transform(Image.open(self.paths[i]).convert('RGB'))


@torch.no_grad()
def compute_embeddings(models, transform, image_paths, batch_size, device):
    per_model = {m: {} for m in models}
    for name, paths in tqdm(image_paths.items(), desc='embed datasets'):
        loader = DataLoader(ImageListDataset(paths, transform), batch_size=batch_size, shuffle=False)
        chunks = {m: [] for m in models}
        for batch in loader:                                            # batch: dict model -> Tensor [B, 3, H, W]
            for m, embedder in models.items():
                chunks[m].append(embedder(batch[m].to(device)).cpu())
        for m in models:
            per_model[m][name] = torch.cat(chunks[m])
    embeddings = {name: combine_embeddings({m: per_model[m][name] for m in models}) for name in image_paths}
    return embeddings, per_model


embeddings, embeddings_per_model = compute_embeddings(embed_models, embed_transform, image_paths, EMBED_BATCH_SIZE, DEVICE)

for name, X in embeddings.items():
    print(f'{name:<30} embeddings {tuple(X.shape)}  vs  {len(image_paths[name])} paths')


embed datasets: 100%|██████████| 2/2 [00:08<00:00,  4.40s/it]

ETs                            embeddings (22, 768)  vs  22 paths
stairs                         embeddings (51, 768)  vs  51 paths


### 2c · Sanity check *(optional)*
Compares two images of the same scene with one image of a different scene. The first similarity should be clearly larger than the second. This cell skips itself automatically when the example images are not on disk (which is the case on the hidden test set).

In [72]:
# ══════════════════════════════════════════════════════════════════════
# STEP 2c · EMBEDDING SANITY CHECK (optional, not used downstream)
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   EXAMPLE_IMAGES    list of 3 Paths: [scene A, scene A (same), scene B]
#   embed_image       from step 2a (combined embedding of the enabled models)
# OUTPUTS
#   sim_same_scene    float, cosine similarity of images 0 and 1  (expected: high)
#   sim_diff_scene    float, cosine similarity of images 0 and 2  (expected: lower)
# ══════════════════════════════════════════════════════════════════════

if all(p.exists() for p in EXAMPLE_IMAGES):
    y = [embed_image(Image.open(p).convert('RGB')) for p in EXAMPLE_IMAGES]   # each y[i]: [1, D_total]

    sim_same_scene = F.cosine_similarity(y[0], y[1]).item()
    sim_diff_scene = F.cosine_similarity(y[0], y[2]).item()
    print(f'same scene:      {sim_same_scene:.4f}')
    print(f'different scene: {sim_diff_scene:.4f}')
else:
    sim_same_scene = sim_diff_scene = None
    print('Step 2c skipped: example images not found (expected on the hidden test set).')


same scene:      0.8959
different scene: -0.0205


---
## Step 3 · Pair shortlisting

**What it does:** for each dataset, L2-normalises the embeddings and finds the `K_NEIGHBORS` most similar images for every image. The union of these neighbour relations (duplicates removed, direction ignored) becomes the list of candidate pairs that will be matched in step 5.

**Why:** matching is the expensive part. With `N` images, all pairs would be `N(N-1)/2`; k-NN limits this to at most `N·k`.

**Replace with:** HDBSCAN clustering followed by all-pairs-within-cluster, a similarity threshold instead of fixed `k`, retrieval with NetVLAD/CLIP, or a combination of several.

**Variants:** `K_NEIGHBORS`, `KNN_METRIC`, `SHORTLIST_MODE` (`'combined'` or `'per_model_union'` with `K_PER_MODEL`), `QUERY_EXPANSION` / `QE_TOP_K` (0.2, *VARIANT SWITCHES*).


In [73]:
# ══════════════════════════════════════════════════════════════════════
# STEP 3 · PAIR SHORTLISTING (k-NN on embeddings)
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   embeddings    dict[str, Tensor[N, D]]   from step 2b (combined embedding)
#   embeddings_per_model   dict[str, dict[str, Tensor[N, D_m]]]   from step 2b (only for SHORTLIST_MODE = 'per_model_union')
#   image_paths   dict[str, list[Path]]     from step 1 (only the file names are used)
#   K_NEIGHBORS, KNN_METRIC, SHORTLIST_MODE, K_PER_MODEL, QUERY_EXPANSION, QE_TOP_K
# OUTPUTS
#   pairs         dict[str, list[tuple[str, str]]]
#                 dataset -> candidate pairs as (file_name_a, file_name_b);
#                 each tuple is sorted alphabetically, no (a, a), no duplicates
#                 size per dataset: at most about N * K_NEIGHBORS ('combined') or N * K_PER_MODEL * #models ('per_model_union')
#   pair_idx      dict[str, list[tuple[int, int]]]
#                 same pairs as row indices (i, j) with i < j into embeddings[name] / image_paths[name]
# ══════════════════════════════════════════════════════════════════════

def knn_pairs(Xs, files, k, metric, qe_top_k=0):
    n = len(files)
    k = min(k, n - 1)                                   # a dataset can be smaller than k
    found = set()
    for X in Xs:                                        # one matrix = plain k-NN; several = union of their k-NN pairs
        X = F.normalize(X, dim=1).cpu().numpy()         # [N, D] float32 numpy, unit length

        if qe_top_k > 0 and n > 1:                      # query expansion: merge each image's top neighbours into its own embedding
            q = min(qe_top_k, n - 1)
            neigh = NearestNeighbors(n_neighbors=q + 1, metric=metric).fit(X).kneighbors(X, return_distance=False)
            X = np.stack([X[[i] + [j for j in row if j != i][:q]].mean(axis=0) for i, row in enumerate(neigh)])
            X = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)

        knn = NearestNeighbors(n_neighbors=k + 1, metric=metric).fit(X)   # +1 because the image finds itself
        _, neigh = knn.kneighbors(X)                                      # [N, k+1] neighbour indices
        found |= {(min(i, j), max(i, j)) for i in range(n) for j in neigh[i] if j != i}
    idx_pairs  = sorted(found)
    file_pairs = [tuple(sorted((files[i], files[j]))) for i, j in idx_pairs]
    return file_pairs, idx_pairs


if KNN_METRIC not in ('cosine', 'euclidean', 'manhattan'):
    raise ValueError(f"KNN_METRIC must be 'cosine', 'euclidean' or 'manhattan', got {KNN_METRIC!r}")
if SHORTLIST_MODE not in ('combined', 'per_model_union'):
    raise ValueError(f"SHORTLIST_MODE must be 'combined' or 'per_model_union', got {SHORTLIST_MODE!r}")

pairs, pair_idx = {}, {}
for name, X in embeddings.items():
    files = [p.name for p in image_paths[name]]
    if SHORTLIST_MODE == 'combined':
        Xs, k = [X], K_NEIGHBORS
    else:
        Xs, k = [embeddings_per_model[m][name] for m in embeddings_per_model], K_PER_MODEL
    pairs[name], pair_idx[name] = knn_pairs(Xs, files, k, KNN_METRIC, QE_TOP_K if QUERY_EXPANSION else 0)

for name in pairs:
    print(f'{name:<30} {len(image_paths[name]):>5} images -> {len(pairs[name]):>6} candidate pairs')
print(f'-> {sum(map(len, pairs.values()))} candidate pairs in total')


ETs                               22 images ->    231 candidate pairs
stairs                            51 images ->   1071 candidate pairs
-> 1302 candidate pairs in total


---
## Step 4 · Local feature extraction (ALIKED or SuperPoint)

**What it does:** detects up to `COARSE_MAX_KEYPOINTS` interest points in every image that appears in at least one candidate pair, and computes a descriptor for each (128-dim for ALIKED, 256-dim for SuperPoint). Features are computed **once per image** and reused for all of its pairs.

The image is internally resized so its longer side is `COARSE_RESIZE`, but the returned keypoint coordinates are mapped back to the **original image resolution**.

### 4a · Load the extractor

In [74]:
# ══════════════════════════════════════════════════════════════════════
# STEP 4a · LOAD LOCAL-FEATURE EXTRACTORS
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   LIGHTGLUE_FEATURES, COARSE_MAX_KEYPOINTS, ALIKED_MODEL_NAME, ALIKED_THRESHOLD, DEVICE
# OUTPUTS
#   extractors    dict[str, object]   feature type -> object with
#                 .extract(img: Tensor[3,H,W] in [0,1], resize=int) -> dict (see step 4b)
# ══════════════════════════════════════════════════════════════════════

def load_extractor(feature, max_keypoints, device):
    if feature == 'aliked':
        return ALIKED(
            model_name=ALIKED_MODEL_NAME,
            max_num_keypoints=ALIKED_MAX_KEYPOINTS,
            detection_threshold=ALIKED_THRESHOLD,
        ).eval().to(device)
    if feature == 'superpoint':
        return SuperPoint(
            max_num_keypoints=max_keypoints
        ).eval().to(device)
    raise ValueError(f'Unknown feature type: {feature}')


extractors = {feature: load_extractor(feature, COARSE_MAX_KEYPOINTS, DEVICE) for feature in LIGHTGLUE_FEATURES}

# Reserved for a later fine pass (FINE_* hyperparameters), currently unused:
# fine_extractor = SuperPoint(max_num_keypoints=FINE_MAX_KEYPOINTS).eval().to(DEVICE)

### 4b · Extract features for all images
Output stays on `DEVICE` so matching needs no transfers. With many images and a large keypoint budget this uses a noticeable amount of GPU memory (roughly `N_images × N_keypoints × descriptor_dim × 4` bytes for descriptors).

In [75]:
# ══════════════════════════════════════════════════════════════════════
# STEP 4b · EXTRACT FEATURES
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   extractors    dict[str, object]   from step 4a
#   pairs         dict[str, list[tuple[str, str]]]   from step 3 (decides which images need features)
#   IMAGE_DIR, COARSE_RESIZE, DEVICE
# OUTPUTS
#   features      dict[str, dict[str, dict[str, dict]]]
#                 features[dataset][file_name][feature_type] = {
#                     'keypoints'       Tensor [1, N, 2]    float32, (x, y) in ORIGINAL image pixels, N <= COARSE_MAX_KEYPOINTS
#                     'keypoint_scores' Tensor [1, N]
#                     'descriptors'     Tensor [1, N, D]    D = 128 for ALIKED, 256 for SuperPoint
#                     'image_size'      Tensor [1, 2]       (width, height) of the ORIGINAL image
#                 }
# ══════════════════════════════════════════════════════════════════════

@torch.no_grad()
def extract_features(extractors, image_dir, files, resize, device, desc=''):
    feats = {}
    for f in tqdm(files, desc=desc, leave=False):
        img = load_image(image_dir / f).to(device)            # Tensor [3, H, W], float in [0, 1]
        feats[f] = {feature: extractor.extract(img, resize=resize) for feature, extractor in extractors.items()}
    return feats


features = {}
for name in pairs:
    files = sorted({f for pair in pairs[name] for f in pair})
    features[name] = extract_features(extractors, IMAGE_DIR / name, files, COARSE_RESIZE, DEVICE, desc=f'extract {name}')

for name, feats in features.items():
    n_kp = [sum(g['keypoints'].shape[1] for g in f.values()) for f in feats.values()]
    if not n_kp:
        print(f'{name:<30}     0 images with candidate pairs (all become outliers)')
        continue
    print(f'{name:<30} {len(feats):>5} images, keypoints/image: min {min(n_kp)}, max {max(n_kp)}')



ETs                               22 images, keypoints/image: min 5185, max 5996
stairs                            51 images, keypoints/image: min 4376, max 5883


---
## Step 5 · Local feature matching (LightGlue)

**What it does:** for each candidate pair, finds which keypoint in image A corresponds to which keypoint in image B by comparing descriptors and spatial context. The result is a *raw* match list: it still contains wrong matches, which are removed in step 6.

Each match record also stores `scale`, the factor between original image size and the resized size used for extraction. Step 6 uses it so the RANSAC pixel threshold means the same thing for small and large images.

**Replace with:** SuperGlue, LoFTR/RoMa (detector-free, would replace steps 4+5 together), MASt3R matching, mutual-nearest-neighbour baseline, ...

### 5a · Load the matcher

In [76]:
# ══════════════════════════════════════════════════════════════════════
# STEP 5a · LOAD MATCHERS
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   LIGHTGLUE_FEATURES, DEVICE
# OUTPUTS
#   matchers      dict[str, callable]   feature type -> callable:
#                 {'image0': feats_a, 'image1': feats_b} -> dict containing
#                 'matches' Tensor [B, M, 2] (int64 index pairs); use rbd() to drop the batch dim
# ══════════════════════════════════════════════════════════════════════

matchers = {
    feature: LightGlue(
        features=feature
    ).eval().to(DEVICE)
    for feature in LIGHTGLUE_FEATURES
}

### 5b · Match all candidate pairs

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 5b · MATCH ALL CANDIDATE PAIRS
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   matchers      dict[str, callable]          from step 5a
#   extractors    dict[str, object]            from step 4a (used for rotated copies and tile crops)
#   features      dict[str, dict[str, dict[str, dict]]]   from step 4b
#   pairs         dict[str, list[tuple[str, str]]]   from step 3
#   IMAGE_DIR, COARSE_RESIZE (also used for `scale`), DEVICE
#   ROTATION_RETRY_BELOW, ROTATION_RETRY_FEATURES, ROTATION_MIN_GAIN
#   TILE_ENABLE, TILE_GRID, TILE_OVERLAP, TILE_RESIZE, TILE_FEATURES, TILE_ABOVE, TILE_WITH_FULL
# OUTPUTS
#   match_records list[dict], one entry per candidate pair, keys:
#                 'dataset'     str
#                 'scene'       None (placeholder, filled by nobody yet)
#                 'pair'        (file_a, file_b)
#                 'keypoints0'  np.float32 [N0, 2]   (x, y) in original pixels of image a: all feature types at 0 deg,
#                                                    then the keypoints of each rotation and each tile of image a used by any of its pairs
#                 'keypoints1'  np.float32 [N1, 2]   (x, y) in original pixels of image b
#                 'matches0'    np.int64   [N0]      for keypoint i of image a: index of the matching
#                                                    keypoint in image b, or -1 if unmatched
#                 'match_scores0' np.float32 [N0]    LightGlue confidence of that match (0 where unmatched); step 6 sorts by it for PROSAC
#                 'scale'       float                max(original image side) / COARSE_RESIZE
#                 'rotation'    int                  degrees image b was rotated by for the full-image matches
#                 'n_tile_matches' int              number of tile-pass matches added to this pair (0 if not tiled)
# ══════════════════════════════════════════════════════════════════════

def rotated_to_original(keypoints, k, width, height):
    # Undoes torch.rot90(image, k, dims=(1, 2)): (x, y) in the rotated image -> (x, y) in the original image.
    x, y = keypoints[:, 0], keypoints[:, 1]
    if k == 1:
        return np.column_stack((width - 1 - y, x))
    if k == 2:
        return np.column_stack((width - 1 - x, height - 1 - y))
    return np.column_stack((y, height - 1 - x))


def tile_boxes(height, width, grid, overlap):
    # (left, top, right, bottom) in original pixels for each crop of a `grid` = (rows, cols) split, grown by `overlap`.
    rows, cols = grid
    boxes = []
    for r in range(rows):
        for c in range(cols):
            top, bottom = r * height / rows, (r + 1) * height / rows
            left, right = c * width / cols, (c + 1) * width / cols
            dy, dx = (bottom - top) * overlap, (right - left) * overlap
            boxes.append((int(max(0, left - dx)), int(max(0, top - dy)),
                          int(min(width, right + dx)), int(min(height, bottom + dy))))
    return boxes


def run_matcher(matcher, feats_a, feats_b):
    out = rbd(matcher({'image0': feats_a, 'image1': feats_b}))
    return out['matches'].cpu().numpy(), out['scores'].cpu().numpy().astype(np.float32)     # [M, 2] index pairs, [M] confidences


@torch.no_grad()
def match_pairs(matchers, extractors, features, pairs, image_dir, resize, device,
                retry_below, retry_features, min_gain,
                tile_enable, tile_grid, tile_overlap, tile_resize, tile_features, tile_above, tile_with_full):
    n_tiles = tile_grid[0] * tile_grid[1]
    records = []
    for name in pairs:
        # Rotated and tile features are only extracted for images that need them, and kept on the CPU to save GPU memory.
        rotated_features, tile_feats = {}, {}

        def get_rotated(file, k):
            if (file, k) not in rotated_features:
                img = torch.rot90(load_image(image_dir / name / file), k, dims=(1, 2)).to(device)
                rotated_features[(file, k)] = {
                    feature: {key: value.cpu() for key, value in extractors[feature].extract(img, resize=resize).items()}
                    for feature in retry_features
                }
            return rotated_features[(file, k)]

        def get_tiles(file):
            # Each image is cropped once and reused across all of its pairs. Keeps (feature_dict_cpu, (x0, y0)) per tile.
            if file not in tile_feats:
                img = load_image(image_dir / name / file)
                _, height, width = img.shape
                per = {feature: [] for feature in tile_features}
                for (x0, y0, x1, y1) in tile_boxes(height, width, tile_grid, tile_overlap):
                    crop = img[:, y0:y1, x0:x1].to(device)
                    for feature in tile_features:
                        feat = {key: value.cpu() for key, value in extractors[feature].extract(crop, resize=tile_resize).items()}
                        per[feature].append((feat, (x0, y0)))
                tile_feats[file] = per
            return tile_feats[file]

        # Every image starts with its 0 deg keypoints (one block per matcher). Rotation and tile blocks are appended lazily;
        # `offsets` records where each block starts in the per-image keypoint list. Keys: ('full', k, feature) / ('tile', i, feature).
        keypoints, offsets = {}, {}

        def register_block(file, key, make):
            if file not in keypoints:
                keypoints[file], offsets[file], start = [], {}, 0
                for f in matchers:
                    pts = features[name][file][f]['keypoints'][0].cpu().numpy()   # [N, 2]
                    keypoints[file].append(pts)
                    offsets[file][('full', 0, f)] = start
                    start += len(pts)
            if key not in offsets[file]:
                offsets[file][key] = sum(len(p) for p in keypoints[file])
                keypoints[file].append(make())
            return offsets[file][key]

        def full_offset(file, k, feature):
            if k == 0:
                return register_block(file, ('full', 0, feature), None)         # the 0 deg block always exists
            def make():
                width, height = features[name][file][feature]['image_size'][0].tolist()
                points = get_rotated(file, k)[feature]['keypoints'][0].numpy()
                return rotated_to_original(points, k, width, height)
            return register_block(file, ('full', k, feature), make)

        def tile_offset(file, i, feature):
            def make():
                feat, (x0, y0) = get_tiles(file)[feature][i]
                points = feat['keypoints'][0].numpy().copy()
                points[:, 0] = points[:, 0] + x0                                # crop pixels -> original pixels
                points[:, 1] = points[:, 1] + y0
                return points
            return register_block(file, ('tile', i, feature), make)

        pair_matches = []
        for a, b in tqdm(pairs[name], desc=f'match {name}'):
            idx = {feature: run_matcher(matcher, features[name][a][feature], features[name][b][feature])
                   for feature, matcher in matchers.items()}                # feature -> ([M, 2] matches, [M] scores)
            best_k, best_count = 0, sum(len(m) for m, _ in idx.values())

            if best_count < retry_below:
                needed = max(min_gain * best_count, 1)
                for k in (1, 2, 3):
                    rotated = {feature: run_matcher(matchers[feature],
                                   features[name][a][feature],
                                   {key: value.to(device) for key, value in get_rotated(b, k)[feature].items()})
                               for feature in retry_features}
                    count = sum(len(m) for m, _ in rotated.values())
                    if count >= needed and count > best_count:
                        idx, best_k, best_count = rotated, k, count

            # Tile pass: only on pairs that already match well. Views are the full image (index -1) plus the n_tiles crops;
            # the full x full view is skipped because the coarse pass above already did it. tmatches: (feature, a_key, b_key, [M, 2]).
            tmatches = []
            if tile_enable and best_count >= tile_above:
                tiles_a, tiles_b = get_tiles(a), get_tiles(b)
                for feature in tile_features:
                    a_views = [(('full',), features[name][a][feature])]
                    a_views += [(('tile', i), {key: value.to(device) for key, value in tiles_a[feature][i][0].items()}) for i in range(n_tiles)]
                    b_views = [(('full',), features[name][b][feature])]
                    b_views += [(('tile', j), {key: value.to(device) for key, value in tiles_b[feature][j][0].items()}) for j in range(n_tiles)]
                    for a_key, a_feat in a_views:
                        for b_key, b_feat in b_views:
                            if a_key[0] == 'full' and b_key[0] == 'full':
                                continue
                            if not tile_with_full and (a_key[0] == 'full' or b_key[0] == 'full'):
                                continue
                            m, sc = run_matcher(matchers[feature], a_feat, b_feat)
                            if len(m):
                                tmatches.append((feature, a_key, b_key, m, sc))

            pair_matches.append((a, b, best_k, idx, tmatches))

        # Offsets are only final once every pair is matched, because rotations and tiles extend the keypoint lists.
        pair_indices = []
        for a, b, k, idx, tmatches in pair_matches:
            shifted = [m + [full_offset(a, 0, feature), full_offset(b, k, feature)]        # full-image matches
                       for feature, (m, _) in idx.items()]
            scores = [sc for _, sc in idx.values()]                                        # same order as `shifted`
            for feature, a_key, b_key, m, sc in tmatches:                                  # tile matches
                ao = full_offset(a, 0, feature) if a_key[0] == 'full' else tile_offset(a, a_key[1], feature)
                bo = full_offset(b, 0, feature) if b_key[0] == 'full' else tile_offset(b, b_key[1], feature)
                shifted.append(m + [ao, bo])
                scores.append(sc)
            tcount = int(sum(len(t[3]) for t in tmatches))
            pair_indices.append((a, b, k, np.concatenate(shifted), np.concatenate(scores), tcount))

        merged = {file: np.concatenate(points) for file, points in keypoints.items()}

        for a, b, k, idx, sc, tcount in pair_indices:
            matches0 = np.full(len(merged[a]), -1, dtype=np.int64)
            matches0[idx[:, 0]] = idx[:, 1]                                                # one match per keypoint of a; later wins on collisions
            match_scores0 = np.zeros(len(merged[a]), dtype=np.float32)
            match_scores0[idx[:, 0]] = sc                                                  # same indexing -> same winner as matches0

            fa, fb = next(iter(features[name][a].values())), next(iter(features[name][b].values()))
            records.append({
                'dataset': name, 'scene': None, 'pair': (a, b),
                'keypoints0': merged[a], 'keypoints1': merged[b],
                'matches0': matches0, 'match_scores0': match_scores0,
                'scale': max(fa['image_size'].max().item(), fb['image_size'].max().item()) / resize,
                'rotation': 90 * k,
                'n_tile_matches': tcount,
            })
    return records


match_records = match_pairs(matchers, extractors, features, pairs, IMAGE_DIR, COARSE_RESIZE, DEVICE,
                            ROTATION_RETRY_BELOW, ROTATION_RETRY_FEATURES, ROTATION_MIN_GAIN,
                            TILE_ENABLE, TILE_GRID, TILE_OVERLAP, TILE_RESIZE, TILE_FEATURES, TILE_ABOVE, TILE_WITH_FULL)

n_matches = [int((r['matches0'] >= 0).sum()) for r in match_records]
if n_matches:
    print(f'{len(match_records)} pairs matched; raw matches per pair: '
          f'min {min(n_matches)}, median {int(np.median(n_matches))}, max {max(n_matches)}')
else:
    print('0 pairs matched (no dataset has candidate pairs)')
n_rotated = sum(r['rotation'] != 0 for r in match_records)
print(f'{n_rotated} of {len(match_records)} pairs use a rotated image b')
tiled = [r['n_tile_matches'] for r in match_records if r['n_tile_matches'] > 0]
print(f'{len(tiled)} of {len(match_records)} pairs got tile matches; '
      f'tile matches per tiled pair: median {int(np.median(tiled)) if tiled else 0}')


match stairs:  72%|███████▏  | 773/1071 [15:44<06:21,  1.28s/it]

---
## Step 6 · Geometric verification (RANSAC)

**What it does:** for each pair, fits a *fundamental matrix* to the matched points with RANSAC. Matches that agree with a single consistent two-view geometry are **inliers**; the rest are rejected as wrong matches. The step reports how many matches and inliers each pair has and the inlier ratio.

The pixel threshold is `RANSAC_THRESHOLD × scale`, so it is given in the resized-image scale and converted to original pixels per pair. Pairs with fewer than `RANSAC_MIN_MATCHES` matches get no fundamental matrix and zero inliers.

**Replace with:** MAGSAC++ (`cv2.USAC_MAGSAC`), essential-matrix / homography verification, PoseLib, ...

**Variants:** `GEOM_ALGORITHM` (`ransac`, `lo_ransac`, `magsac++`, `prosac`, `usac`) and `GEOM_MODEL` (`fundamental`, `essential`, `homography`) in 0.2 (*VARIANT SWITCHES*). Unsupported combinations fall back to RANSAC with a printed warning.


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 6 · GEOMETRIC VERIFICATION
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   match_records  list[dict]   from step 5b ('match_scores0' is needed for GEOM_ALGORITHM = 'prosac')
#   GEOM_ALGORITHM, GEOM_MODEL, GEOM_FOCAL_FACTOR   (VARIANT SWITCHES)
#   RANSAC_THRESHOLD, RANSAC_CONFIDENCE, RANSAC_MIN_MATCHES, VERIFIED_MIN_INLIERS
#   IMAGE_DIR      only for GEOM_MODEL = 'essential' (image size -> guessed intrinsics)
# OUTPUTS
#   ransac_results list[dict], same order as match_records. Every input key is kept, plus:
#                  'fundamental_matrix'  np.ndarray [3, 3] float64, or None if not estimated
#                                        (essential: derived from E and the guessed intrinsics; homography: always None)
#                  'essential_matrix'    np.ndarray [3, 3] or None   (only with GEOM_MODEL = 'essential')
#                  'homography_matrix'   np.ndarray [3, 3] or None   (only with GEOM_MODEL = 'homography')
#                  'geom_model'          str     model used
#                  'geom_algorithm'      str     algorithm used (after any fallback to 'ransac')
#                  'inlier_mask'         np.ndarray [M] bool   over the M matched keypoint pairs
#                  'inlier_matches'      np.ndarray [K, 2] int64   K = #inliers; (idx in keypoints0, idx in keypoints1)
#                  'num_matches'         int     M
#                  'num_inliers'         int     K
#                  'inlier_ratio'        float   K / M  (0.0 if M == 0)
#                  'verified'            bool    K >= VERIFIED_MIN_INLIERS
# ══════════════════════════════════════════════════════════════════════

GEOM_ALGORITHMS = ('ransac', 'lo_ransac', 'magsac++', 'prosac', 'usac')
GEOM_MODELS = ('fundamental', 'essential', 'homography')
USAC_FLAGS = {'lo_ransac': 'USAC_DEFAULT',      # LO-RANSAC
              'magsac++': 'USAC_MAGSAC',
              'prosac':   'USAC_PROSAC',        # samples from the top of the list: matches must be sorted best -> worst
              'usac':     'USAC_ACCURATE'}      # accurate USAC preset (GC-RANSAC)


def cv_method(algorithm, model):
    '''OpenCV method flag for (algorithm, model), or None if this OpenCV build does not offer it.'''
    if algorithm == 'ransac':
        return cv2.FM_RANSAC if model == 'fundamental' else cv2.RANSAC
    return getattr(cv2, USAC_FLAGS[algorithm], None)


_image_sizes = {}


def guess_intrinsics(image_dir, dataset, file, focal_factor):
    '''(principal point [2], focal) guessed from the image size: image centre, f = focal_factor * max(w, h).'''
    if (dataset, file) not in _image_sizes:
        _, height, width = load_image(image_dir / dataset / file).shape      # same loader as steps 4b/5b -> same pixel frame
        _image_sizes[(dataset, file)] = (width, height)
    width, height = _image_sizes[(dataset, file)]
    return np.array([width / 2, height / 2]), focal_factor * max(width, height)


def fit_geometry(model, method, pts0, pts1, threshold, confidence, intr0=None, intr1=None):
    '''One OpenCV call. Returns (3x3 matrix, bool inlier mask [M]), or (None, None) if no model was found.'''
    if model == 'fundamental':
        mat, mask = cv2.findFundamentalMat(pts0, pts1, method=method,
                                           ransacReprojThreshold=threshold, confidence=confidence)
    elif model == 'homography':
        mat, mask = cv2.findHomography(pts0, pts1, method=method,
                                       ransacReprojThreshold=threshold, confidence=confidence)
    else:   # essential: run on normalised coordinates so each image can have its own intrinsics
        (c0, f0), (c1, f1) = intr0, intr1
        mat, mask = cv2.findEssentialMat((pts0 - c0) / f0, (pts1 - c1) / f1, cameraMatrix=np.eye(3), method=method,
                                         prob=confidence, threshold=threshold / ((f0 + f1) / 2))
    if mat is None or mask is None or mat.shape != (3, 3):
        return None, None
    return mat, mask.ravel().astype(bool)


def verify_pairs(match_records, algorithm, model, threshold, confidence, min_matches,
                 verified_min_inliers, image_dir, focal_factor, report_every=500):
    if algorithm not in GEOM_ALGORITHMS:
        raise ValueError(f'GEOM_ALGORITHM must be one of {GEOM_ALGORITHMS}, got {algorithm!r}')
    if model not in GEOM_MODELS:
        raise ValueError(f'GEOM_MODEL must be one of {GEOM_MODELS}, got {model!r}')

    def fall_back(reason):
        nonlocal algorithm, method
        print(f"WARNING: {reason} -> falling back to RANSAC ({model})", flush=True)
        algorithm, method = 'ransac', cv_method('ransac', model)

    method = cv_method(algorithm, model)
    if method is None:
        fall_back(f'cv2.{USAC_FLAGS[algorithm]} is not available in OpenCV {cv2.__version__}')
    if algorithm == 'prosac' and not all('match_scores0' in r for r in match_records):
        fall_back("'prosac' needs 'match_scores0' in the match records (re-run step 5b)")

    results = []
    start_time = time.perf_counter()

    for i, rec in enumerate(match_records):
        kp0, kp1, matches0 = rec['keypoints0'], rec['keypoints1'], rec['matches0']

        matched_indices0 = np.flatnonzero(matches0 >= 0)
        matched_indices1 = matches0[matched_indices0]

        pts0 = kp0[matched_indices0].astype(np.float32)
        pts1 = kp1[matched_indices1].astype(np.float32)
        num_matches = len(pts0)

        matrix, inliers, intr = None, np.zeros(num_matches, dtype=bool), None

        if num_matches >= min_matches:
            # PROSAC needs the matches best-first (LightGlue score, descending); the mask is mapped back to the original order below.
            order = (np.argsort(-rec['match_scores0'][matched_indices0], kind='stable')
                     if algorithm == 'prosac' else np.arange(num_matches))
            if model == 'essential':
                intr = [guess_intrinsics(image_dir, rec['dataset'], f, focal_factor) for f in rec['pair']]
            args = (pts0[order], pts1[order], threshold * rec['scale'], confidence, *(intr or (None, None)))
            try:
                mat, mask = fit_geometry(model, method, *args)
            except cv2.error as exc:
                if algorithm == 'ransac':
                    raise
                fall_back(f"{algorithm} failed for model {model} ({str(exc).strip().splitlines()[-1][:100]})")
                mat, mask = fit_geometry(model, method, *args)
            if mat is not None:
                matrix = mat
                inliers[order] = mask

        F_matrix = E_matrix = H_matrix = None
        if matrix is not None:
            if model == 'fundamental':
                F_matrix = matrix
            elif model == 'homography':
                H_matrix = matrix
            else:
                E_matrix = matrix
                k_inv = [np.array([[1 / f, 0, -c[0] / f], [0, 1 / f, -c[1] / f], [0, 0, 1]]) for c, f in intr]
                F_matrix = k_inv[1].T @ E_matrix @ k_inv[0]          # F = K1^-T E K0^-1

        num_inliers = int(inliers.sum())

        results.append({
            **rec,
            'fundamental_matrix': F_matrix,
            'essential_matrix': E_matrix,
            'homography_matrix': H_matrix,
            'geom_model': model,
            'geom_algorithm': algorithm,
            'inlier_mask': inliers,
            'inlier_matches': np.column_stack(
                (matched_indices0[inliers], matched_indices1[inliers])
            ),
            'num_matches': num_matches,
            'num_inliers': num_inliers,
            'inlier_ratio': num_inliers / num_matches if num_matches else 0.0,
            'verified': num_inliers >= verified_min_inliers,
        })

        if (i + 1) % report_every == 0 or (i + 1) == len(match_records):
            elapsed = time.perf_counter() - start_time
            avg_time = elapsed / (i + 1)
            remaining = avg_time * (len(match_records) - i - 1)

            print(
                f"Processed {i + 1:,}/{len(match_records):,} pairs "
                f"({(i + 1) / len(match_records):.1%}) | "
                f"Verified: {sum(r['verified'] for r in results):,} | "
                f"Elapsed: {elapsed / 60:.1f} min | "
                f"ETA: {remaining / 60:.1f} min",
                flush=True
            )

    return results


ransac_results = verify_pairs(match_records, GEOM_ALGORITHM, GEOM_MODEL, RANSAC_THRESHOLD, RANSAC_CONFIDENCE,
                              RANSAC_MIN_MATCHES, VERIFIED_MIN_INLIERS, IMAGE_DIR, GEOM_FOCAL_FACTOR)

used = ransac_results[-1]['geom_algorithm'] if ransac_results else GEOM_ALGORITHM
print(f'Geometric verification: {GEOM_MODEL} model, {used} (requested: {GEOM_ALGORITHM})')
print(f'Processed {len(ransac_results)} image pairs, {sum(r["verified"] for r in ransac_results)} verified.')
for r in ransac_results[:10]:
    print(f"{r['dataset']} | {r['pair'][0]} <-> {r['pair'][1]} | "
          f"matches: {r['num_matches']} | inliers: {r['num_inliers']} | ratio: {r['inlier_ratio']:.2%}")


---
## Step 7 · Pair filtering

**What it does:** keeps only pairs that are trustworthy enough to feed into SfM. A pair must pass **all three** tests:

* at least `MIN_MATCHES` raw matches,
* at least `MIN_INLIERS` RANSAC inliers,
* an inlier ratio of at least `MIN_INLIER_RATIO`.

Stricter thresholds give fewer but cleaner pairs (fewer wrong registrations, but more images may end up as outliers); looser thresholds do the opposite. This is one of the most useful knobs to sweep.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 7 · FILTER PAIRS
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   ransac_results  list[dict]   from step 6
#   MIN_MATCHES, MIN_INLIERS, MIN_INLIER_RATIO
# OUTPUTS
#   filtered_pairs  list[dict]   subset of ransac_results (same dict structure, same relative order)
# ══════════════════════════════════════════════════════════════════════

def filter_pairs(results, min_matches, min_inliers, min_inlier_ratio):
    return [
        r for r in results
        if r['num_matches'] >= min_matches
        and r['num_inliers'] >= min_inliers
        and r['inlier_ratio'] >= min_inlier_ratio
    ]


filtered_pairs = filter_pairs(ransac_results, MIN_MATCHES, MIN_INLIERS, MIN_INLIER_RATIO)

print(f'Total pairs before filtering: {len(ransac_results)}')
print(f'Pairs retained: {len(filtered_pairs)}')
print(f'Pairs rejected: {len(ransac_results) - len(filtered_pairs)}')
if ransac_results:
    print(f'Retention rate: {len(filtered_pairs) / len(ransac_results):.1%}')

print('\nTop retained pairs:')
for r in sorted(filtered_pairs, key=lambda r: (r['num_inliers'], r['inlier_ratio']), reverse=True)[:15]:
    print(f"{r['dataset']} | {r['pair'][0]} <-> {r['pair'][1]} | "
          f"Matches: {r['num_matches']} | Inliers: {r['num_inliers']} | Ratio: {r['inlier_ratio']:.1%}")

---
## Step 8 · Build the image graph

**What it does:** packs everything COLMAP needs into one `networkx` graph.

* **Nodes** are images `(dataset, file_name)` and carry that image's keypoints.
* **Edges** are the surviving pairs and carry the RANSAC-inlier matches (as keypoint index pairs).

Images that were in a candidate pair but lost all their edges in step 7 are still added as nodes **without keypoints** (`keypoints=None`). They are skipped by COLMAP and end up as `outliers`. Connected components of this graph are the natural candidates for separate scenes.

The step also checks that an image has identical keypoints in every pair it appears in; otherwise match indices would point to different points in different pairs.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 8 · BUILD IMAGE GRAPH
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   filtered_pairs  list[dict]   from step 7
#   pairs           dict[str, list[tuple[str, str]]]   from step 3 (to add images with no surviving edge)
#   KEYPOINT_CONSISTENCY_ATOL
# OUTPUTS
#   image_graph     nx.Graph
#       node key    (dataset: str, file_name: str)
#       node attrs  dataset: str, scene: None, image_name: str,
#                   keypoints: np.float32 [N, 2] (x, y, original pixels)   or None if the image has no edge
#       edge attrs  image0, image1        node keys, define the order of the two match columns
#                   num_matches, num_inliers  int
#                   inlier_ratio              float
#                   fundamental_matrix        np.ndarray [3, 3] or None
#                   inlier_matches            np.uint32 [K, 2]   (idx into image0 keypoints, idx into image1 keypoints)
# ══════════════════════════════════════════════════════════════════════

def build_image_graph(filtered_pairs, pairs, atol):
    graph = nx.Graph()

    for result in filtered_pairs:
        dataset = result['dataset']
        scene = result['scene']
        image_a, image_b = result['pair']
        node_a, node_b = (dataset, image_a), (dataset, image_b)

        kp0 = np.asarray(result['keypoints0'], dtype=np.float32)
        kp1 = np.asarray(result['keypoints1'], dtype=np.float32)
        matches0 = np.asarray(result['matches0'], dtype=np.int32)
        inlier_mask = np.asarray(result['inlier_mask'], dtype=bool)

        # Recover the original keypoint indices of all raw matches.
        matched_indices0 = np.flatnonzero(matches0 >= 0)
        matched_indices1 = matches0[matched_indices0]

        if len(inlier_mask) != len(matched_indices0):
            raise ValueError(f'Inlier mask mismatch for pair {(image_a, image_b)}')

        # Keep only RANSAC inliers, expressed as original keypoint indices.
        inlier_matches = np.column_stack((
            matched_indices0[inlier_mask],
            matched_indices1[inlier_mask],
        )).astype(np.uint32)

        # Store each image's keypoints once; verify they are identical across pairs.
        for node, keypoints_array in ((node_a, kp0), (node_b, kp1)):
            if node not in graph:
                graph.add_node(node, dataset=dataset, scene=scene,
                               image_name=node[1], keypoints=keypoints_array)
            else:
                existing = graph.nodes[node]['keypoints']
                if existing.shape != keypoints_array.shape or not np.allclose(existing, keypoints_array, atol=atol):
                    raise ValueError(f'Inconsistent keypoints for image {node}. '
                                     'Check feature caching and image ordering.')

        # inlier_matches[:, 0] indexes keypoints of image0 (= image_a), [:, 1] those of image1 (= image_b).
        graph.add_edge(
            node_a, node_b,
            image0=node_a, image1=node_b,
            num_matches=int(result['num_matches']),
            num_inliers=int(result['num_inliers']),
            inlier_ratio=float(result['inlier_ratio']),
            fundamental_matrix=result['fundamental_matrix'],
            inlier_matches=inlier_matches,
        )

    # Images that were shortlisted but have no verified edge: add as keypoint-less nodes.
    for name in pairs:
        for f in {f for pair in pairs[name] for f in pair}:
            if (name, f) not in graph:
                graph.add_node((name, f), dataset=name, scene=None, image_name=f, keypoints=None)

    return graph


image_graph = build_image_graph(filtered_pairs, pairs, KEYPOINT_CONSISTENCY_ATOL)

print('COLMAP-ready image graph built.')
print(f'Images (nodes): {image_graph.number_of_nodes()}')
print(f'Verified pairs (edges): {image_graph.number_of_edges()}')
print(f'Connected components: {nx.number_connected_components(image_graph)}')

---
## Step 9 · Inspect connected components *(optional)*

**What it does:** lists the connected components of the image graph, largest first. Each component is a group of images linked by verified matches, so it is a good first guess for "one scene". Useful for judging whether the filter thresholds split or merge scenes.

> The reconstruction in step 10 does **not** use this output; COLMAP finds its own models. This cell is for inspection only (and a starting point if you later want to reconstruct per component).

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 9 · CONNECTED COMPONENTS (inspection only)
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   image_graph   nx.Graph   from step 8
# OUTPUTS
#   components    list[set[tuple[str, str]]]
#                 each set = node keys (dataset, file_name) of one component, sorted by size, largest first
#                 isolated images appear as components of size 1
# ══════════════════════════════════════════════════════════════════════

components = sorted(nx.connected_components(image_graph), key=len, reverse=True)

print('Connected components:', len(components))
print('Sizes of the 10 largest:', [len(c) for c in components[:10]])
for i, component in enumerate(components[:10], start=1):
    datasets = sorted({d for d, _ in component})
    print(f'  Component {i}: {len(component)} images, datasets: {datasets}')

---
## Step 10 · Reconstruction with COLMAP (pycolmap)

Structure-from-Motion estimates a 3D point cloud and a camera pose for each image from the verified matches. It runs **separately for every dataset** and is split into four cells so each part can be replaced on its own:

| Cell | Does | Replace with |
|------|------|--------------|
| 10a | build mapper options | different COLMAP settings |
| 10b | write keypoints and matches into a COLMAP database | — |
| 10c | run incremental mapping | GLOMAP (global SfM), hloc pipeline, ... |
| 10d | convert reconstructions into pose rows | — |

Each model COLMAP returns is treated as one **scene cluster**. Images that are in no model are labelled **outliers**.

### 10a · Mapper options

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 10a · MAPPER OPTIONS
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   MAPPER_MIN_MODEL_SIZE, MAPPER_INIT_MIN_NUM_INLIERS, MAPPER_ABS_POSE_MIN_NUM_INLIERS
# OUTPUTS
#   mapper_options   pycolmap.IncrementalPipelineOptions
# ══════════════════════════════════════════════════════════════════════

mapper_options = pycolmap.IncrementalPipelineOptions()
mapper_options.min_model_size = MAPPER_MIN_MODEL_SIZE
mapper_options.mapper.init_min_num_inliers = MAPPER_INIT_MIN_NUM_INLIERS
mapper_options.mapper.abs_pose_min_num_inliers = MAPPER_ABS_POSE_MIN_NUM_INLIERS

### 10b · Build one COLMAP database per dataset
Creates `WORK_DIR/<dataset>/database.db`, registers the images (camera intrinsics are guessed from the files), and writes in our own keypoints, matches and two-view geometries, so COLMAP does **not** run its own feature extraction or matching. Existing work folders for a dataset are deleted first.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 10b · WRITE COLMAP DATABASES
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   image_graph      nx.Graph   from step 8
#   pairs            dict       from step 3 (only its keys = dataset names are used)
#   IMAGE_DIR, WORK_DIR, COLMAP_CAMERA_MODE, COLMAP_KEYPOINT_OFFSET
# OUTPUTS
#   db_paths         dict[str, Path]
#                    dataset name -> WORK_DIR/<dataset>/database.db  (datasets with no verified pair, or whose build failed, are left out)
#                    containing, for every image that has keypoints:
#                      keypoints            float32 [N, 2], shifted by COLMAP_KEYPOINT_OFFSET
#                      matches              uint32  [K, 2]   per verified pair (RANSAC inliers)
#                      two-view geometry    same inlier matches, config UNCALIBRATED
# ══════════════════════════════════════════════════════════════════════

def build_colmap_database(name, graph, image_dir, work_dir,
                          camera_mode, keypoint_offset):
    start = time.perf_counter()

    nodes = [n for n in graph if n[0] == name and graph.nodes[n]['keypoints'] is not None]
    if not nodes:        # no verified pair at all -> nothing to reconstruct, every image is an outlier
        print(f'[{name}] no verified pairs, skipping COLMAP (all images become outliers)')
        return None

    work = work_dir / name
    shutil.rmtree(work, ignore_errors=True)
    work.mkdir(parents=True)
    db_path = work / 'database.db'
    pycolmap.Database.open(db_path).close()

    pycolmap.import_images(db_path, image_dir / name, camera_mode,
                           image_names=sorted(n[1] for n in nodes))

    db = pycolmap.Database.open(db_path)
    image_id = {img.name: img.image_id for img in db.read_all_images()}

    for node in nodes:
        kps = graph.nodes[node]['keypoints'].astype(np.float32) + keypoint_offset
        db.write_keypoints(image_id[node[1]], kps)

    edges = [(u, v, e) for u, v, e in graph.edges(data=True) if u[0] == name]

    for i, (u, v, edge) in enumerate(edges):
        a, b = edge['image0'][1], edge['image1'][1]
        m = edge['inlier_matches'].astype(np.uint32)
        db.write_matches(image_id[a], image_id[b], m)

        tvg = pycolmap.TwoViewGeometry()
        tvg.inlier_matches = m
        tvg.config = pycolmap.TwoViewGeometryConfiguration.UNCALIBRATED
        db.write_two_view_geometry(image_id[a], image_id[b], tvg)

        if (i + 1) % 500 == 0 or i + 1 == len(edges):
            print(f'[{name}] {i+1}/{len(edges)} pairs | '
                  f'{time.perf_counter()-start:.1f}s elapsed', flush=True)

    db.close()
    return db_path


db_paths = {}
for name in image_paths:
    try:
        db_path = build_colmap_database(name, image_graph, IMAGE_DIR, WORK_DIR,
                                        COLMAP_CAMERA_MODE, COLMAP_KEYPOINT_OFFSET)
    except Exception:
        print(f'[{name}] COLMAP database build FAILED -> all its images become outliers')
        traceback.print_exc()
        continue
    if db_path is not None:
        db_paths[name] = db_path

for name, p in db_paths.items():
    print(f'{name:<30} {p}')
print(f'-> {len(db_paths)}/{len(image_paths)} datasets have a COLMAP database')

### 10c · Incremental mapping
Runs COLMAP's incremental SfM on each database. It may return **several disconnected models** per dataset; their keys (`0, 1, 2, ...`) become the scene names `cluster0`, `cluster1`, ... in step 10d.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 10c · INCREMENTAL MAPPING
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   db_paths         dict[str, Path]   from step 10b
#   mapper_options   pycolmap.IncrementalPipelineOptions   from step 10a
#   IMAGE_DIR, WORK_DIR
# OUTPUTS
#   reconstructions  dict[str, dict[int, pycolmap.Reconstruction]]
#                    dataset -> {model_id: reconstruction}; may be an empty dict if no model could be built.
#                    Every dataset of image_paths has a key; {} also means 'mapping failed or was skipped'.
#                    Models are also written to WORK_DIR/<dataset>/sparse/<model_id>/
#                    Useful accessors: rec.num_reg_images(), rec.images, rec.points3D
# ══════════════════════════════════════════════════════════════════════

def run_incremental_mapping(db_path, image_dir, output_dir, options):
    return pycolmap.incremental_mapping(db_path, image_dir, output_dir, options)


reconstructions = {name: {} for name in image_paths}     # {} = no model for that dataset
for name in db_paths:
    try:
        reconstructions[name] = run_incremental_mapping(db_paths[name], IMAGE_DIR / name,
                                                        WORK_DIR / name / 'sparse', mapper_options)
    except Exception:
        print(f'[{name}] incremental mapping FAILED -> all its images become outliers')
        traceback.print_exc()
        reconstructions[name] = {}

for name, recs in reconstructions.items():
    sizes = sorted((r.num_reg_images() for r in recs.values()), reverse=True)
    print(f'{name:<30} {len(recs)} models, registered images per model: {sizes}')

### 10d · Collect poses
Turns the COLMAP models into the rows needed for the submission. Each model becomes the scene `cluster<model_id>` (COLMAP's own model id); rows are emitted largest model first. Every image that is registered in no model becomes an `outliers` row without a pose.

**Two rules that keep the submission valid:** COLMAP can register the same image in several models, but an image must appear exactly once, so it is kept in the **largest** model only. And the set of images is taken from `image_paths` (everything on disk), not from the candidate pairs, so an image that never entered a pair still gets an `outliers` row. At serialization, each `(dataset, image)` row receives its exact competition-defined `image_id` from `sample_submission.csv`.

**Pose convention:** `R`, `t` come from `image.cam_from_world()`, i.e. they map world to camera coordinates (`x_cam = R · x_world + t`). This is unchanged from the original notebook.


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 10d · COLLECT POSES
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   reconstructions  dict[str, dict[int, pycolmap.Reconstruction]]   from step 10c ({} = no model for that dataset)
#   image_paths      dict[str, list[Path]]   from step 1 (defines the set of images that must appear in the output)
# OUTPUTS
#   rows             list[tuple]  exactly one entry per image on disk:
#                    (dataset: str,
#                     scene:   str   'cluster<model_id>' or 'outliers',
#                     image:   str   file name,
#                     R:       np.ndarray [3, 3] float64 (world->camera rotation)   or None for outliers,
#                     t:       np.ndarray [3]    float64 (world->camera translation) or None for outliers)
# ══════════════════════════════════════════════════════════════════════

def collect_poses(name, recs, all_images):
    rows, registered = [], set()
    for k, rec in sorted(recs.items(), key=lambda kv: -kv[1].num_reg_images()):
        for img in rec.images.values():
            # COLMAP may register the same image in several models. An image must appear
            # exactly once in the submission, so keep it in the largest model only.
            if not img.has_pose or img.name in registered:
                continue
            pose = img.cam_from_world()
            rows.append((name, f'cluster{k}', img.name, pose.rotation.matrix(), pose.translation))
            registered.add(img.name)
    rows += [(name, 'outliers', f, None, None) for f in sorted(all_images - registered)]
    return rows, registered


rows = []
for name in image_paths:
    all_images = {p.name for p in image_paths[name]}          # every image on disk, not only those in a pair
    recs = reconstructions.get(name, {})
    try:
        ds_rows, registered = collect_poses(name, recs, all_images)
    except Exception:
        print(f'[{name}] collecting poses FAILED -> all its images become outliers')
        traceback.print_exc()
        ds_rows, registered = [(name, 'outliers', f, None, None) for f in sorted(all_images)], set()
    rows += ds_rows
    print(f'{name}: {len(recs)} models, {len(registered)}/{len(all_images)} images registered')

---
## Step 11 · Create the submission file

**What it does:** converts the pose rows into the competition's CSV format. The first column, `image_id`, is copied exactly from Kaggle's `sample_submission.csv` using the matching `(dataset, image)` pair; it is never generated from an index or a filename. Rotation matrices and translation vectors are flattened to `;`-separated strings with 9 decimals; outliers get `nan` values.

**Validation:** `validate_and_repair` runs before the file is written. It guarantees that every row in the sample submission appears exactly once, with its original `image_id`; it drops unknown images and adds missing expected images as `outliers`. The final output retains the sample submission's row order.


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 11 · BUILD + SAVE SUBMISSION
# ──────────────────────────────────────────────────────────────────────
# INPUTS
#   rows                     list[tuple]   from step 10d
#   sample_submission_index  pd.DataFrame[image_id, dataset, image] from Step 1
#   SUBMISSION_CSV           Path
# OUTPUTS
#   submission_df            pd.DataFrame, columns (all str):
#                            image_id             exact value from sample_submission.csv
#                            dataset, scene, image
#                            rotation_matrix      'r11;r12;r13;r21;...;r33' (9 values, row-major)
#                                                 or 'nan;nan;...' x9
#                            translation_vector   't1;t2;t3' (3 values) or 'nan;nan;nan'
#                            exactly one row for each sample-submission image
#   (file)                   SUBMISSION_CSV written without the pandas index column
# ══════════════════════════════════════════════════════════════════════

POSE_COLUMNS = ['dataset', 'scene', 'image', 'rotation_matrix', 'translation_vector']
SUBMISSION_COLUMNS = ['image_id', *POSE_COLUMNS]


def format_values(x, n):
    '''np array / None -> n semicolon-separated values, with strict shape checks.'''
    if x is None:
        return ';'.join(['nan'] * n)

    values = np.asarray(x, dtype=np.float64).reshape(-1)
    if len(values) != n:
        raise ValueError(f'Expected {n} pose values, got {len(values)}')
    if not np.isfinite(values).all():
        raise ValueError('A registered pose contains NaN or infinity')
    return ';'.join(f'{v:.9f}' for v in values)


def sort_outliers_last(df):
    return df.sort_values('scene', key=lambda s: s == 'outliers', kind='stable').reset_index(drop=True)


def build_submission(rows):
    '''Serialize the inferred poses; Kaggle IDs are attached only from the template.'''
    df = pd.DataFrame(
        [(dataset, scene, image, format_values(R, 9), format_values(t, 3))
         for dataset, scene, image, R, t in rows],
        columns=POSE_COLUMNS,
    )
    # This order lets duplicate repair retain a registered row before an outlier row.
    return sort_outliers_last(df)


def _valid_serialized_pose(value, n):
    '''Accept exactly n finite values, or n literal NaNs for an unregistered pose.'''
    parts = str(value).split(';')
    if len(parts) != n or any(part == '' for part in parts):
        return False
    try:
        values = np.asarray([float(part) for part in parts], dtype=np.float64)
    except ValueError:
        return False
    return np.isfinite(values).all() or np.isnan(values).all()


def validate_and_repair(predictions, template):
    '''Join predictions to Kaggle's template, preserving its exact IDs and row order.

    - duplicate (dataset, image) predictions are dropped (first wins)
    - predictions absent from the template are dropped
    - template images missing from predictions are added as outliers
    - image_id comes only from the template, never from a generated index
    '''
    if list(predictions.columns) != POSE_COLUMNS:
        raise ValueError(f'Incorrect pose columns: {list(predictions.columns)}')
    if list(template.columns) != ['image_id', 'dataset', 'image']:
        raise ValueError(f'Incorrect sample-submission columns: {list(template.columns)}')

    template = template.copy()
    expected = set(zip(template['dataset'], template['image']))

    n_dup = int(predictions.duplicated(['dataset', 'image']).sum())
    if n_dup:
        print(f'WARNING: dropped {n_dup} duplicate (dataset, image) rows')
        predictions = predictions.drop_duplicates(['dataset', 'image'], keep='first').copy()

    known = [(dataset, image) in expected
             for dataset, image in zip(predictions['dataset'], predictions['image'])]
    if not all(known):
        print(f'WARNING: dropped {len(known) - sum(known)} rows for images absent from sample_submission.csv')
        predictions = predictions.loc[known].copy()

    missing = sorted(expected - set(zip(predictions['dataset'], predictions['image'])))
    if missing:
        print(f'WARNING: added {len(missing)} missing sample-submission images as outliers')
        extra = pd.DataFrame(
            [(dataset, 'outliers', image, format_values(None, 9), format_values(None, 3))
             for dataset, image in missing],
            columns=POSE_COLUMNS,
        )
        predictions = pd.concat([predictions, extra], ignore_index=True)

    # A left merge keeps the sample template's canonical image_id values and order.
    submission = template.merge(
        predictions,
        on=['dataset', 'image'],
        how='left',
        sort=False,
        validate='one_to_one',
    ).loc[:, SUBMISSION_COLUMNS]

    # Fail loudly if a later change ever weakens the template guarantees.
    if len(submission) != len(template):
        raise AssertionError(f'Expected {len(template)} rows, found {len(submission)}')
    if submission[['image_id', 'dataset', 'scene', 'image', 'rotation_matrix', 'translation_vector']].isna().any().any():
        raise AssertionError('Submission contains an empty value')
    if (submission['image_id'].astype(str).str.len() == 0).any():
        raise AssertionError('Submission contains an empty image_id')
    if submission.duplicated(['dataset', 'image']).any():
        raise AssertionError('Submission has duplicate (dataset, image) rows')
    if submission['image_id'].duplicated().any():
        raise AssertionError('Submission has duplicate image_id values')
    if list(submission['image_id']) != list(template['image_id']):
        raise AssertionError('Submission image_id values do not exactly match sample_submission.csv')
    if list(zip(submission['dataset'], submission['image'])) != list(zip(template['dataset'], template['image'])):
        raise AssertionError('Submission row order no longer matches sample_submission.csv')
    if not submission['rotation_matrix'].map(lambda value: _valid_serialized_pose(value, 9)).all():
        raise AssertionError('Submission contains an invalid rotation_matrix serialization')
    if not submission['translation_vector'].map(lambda value: _valid_serialized_pose(value, 3)).all():
        raise AssertionError('Submission contains an invalid translation_vector serialization')

    return submission


submission_df = validate_and_repair(
    build_submission(rows),
    sample_submission_index,
)
SUBMISSION_CSV.parent.mkdir(parents=True, exist_ok=True)
submission_df.to_csv(SUBMISSION_CSV, index=False)

print(f'{len(submission_df)} rows written to {SUBMISSION_CSV} '
      f'({len(sample_submission_index)} exact image_id values, '
      f'{submission_df["dataset"].nunique()} datasets)')
print(submission_df['scene'].value_counts().head(10))


---
## Cleanup

Removes the scratch folders (COLMAP databases and models, copied checkpoints) so that `submission.csv` is the only output of the run.


In [ ]:
shutil.rmtree(WORK_DIR, ignore_errors=True)
shutil.rmtree(TORCH_HOME, ignore_errors=True)
print('Output files:', sorted(p.name for p in OUTPUT_DIR.iterdir()))